# Symbiotic Star: Imaging + Parameter Fitting

**Roche-elongated giant in a binary system**

This notebook is designed to be run with real OIFITS data from a symbiotic star.
It handles:

1. **Roche geometry** — tidal distortion of the giant component
2. **Surface brightness reconstruction** — temperature map of the giant
3. **Binary parameter fitting** — constrain `q`, `a`, `inclination`, `fill factor`
4. **Multi-epoch** — use multiple orbital phases to break degeneracies

---

### Typical symbiotic stars (for scale reference)

| System | Period (d) | a (mas) | Giant radius (mas) | Fill factor |
|--------|-----------|---------|-------------------|-------------|
| R Aqr  | 44 yr     | ~200    | ~30               | ~0.6        |
| Mira AB| 498       | 460     | 40-60             | ~0.5        |
| CH Cyg | 756       | ~12     | ~5                | ~0.8        |
| EG And | 481       | ~3      | ~1.5              | ~0.7        |

---

## 0. Setup

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
from jax import vmap
import scipy.optimize as so
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.insert(0, str(Path.cwd()))  # ensure rotir_jax is importable

from rotir_jax.io.oifits_reader import read_oifits
from rotir_jax.tessellation.healpix import tessellation_healpix
from rotir_jax.geometry.base import (
    create_star, rotation_matrix, apply_rotation, visible_mask
)
from rotir_jax.geometry.roche import (
    compute_roche_shape, eggleton_roche_radius,
    compute_fillout_factor, compute_L1_distance
)
from rotir_jax.geometry.orbits import (
    binary_orbit_absolute, compute_separation, compute_true_anomaly
)
from rotir_jax.forward_model.observables import compute_chi2, compute_observables
from rotir_jax.reconstruction.optimizer import (
    StellarImageReconstructor, reconstruct_stellar_surface, OptimizationResult
)
from rotir_jax.datatypes import OIData, Star

print(f'JAX version: {jax.__version__}')
print(f'JAX default backend: {jax.default_backend()}')
print('Imports OK')

---
## 1. YOUR SYSTEM PARAMETERS

Edit the cells below to describe your symbiotic star.  
If you don't know a parameter well, leave it at the default — you'll fit for it in Section 4.

In [ ]:
# ============================================================
#  DATA INPUT
# ============================================================

# Path to your OIFITS file (or list of files for multi-epoch)
OIFITS_FILE = 'path/to/your/symbiotic_star.fits'   # <-- EDIT THIS

# Wavelength filter (None = use all wavelengths)
WAVE_MIN = None   # e.g. 1.5e-6  (meters)  <-- EDIT (or leave None)
WAVE_MAX = None   # e.g. 1.8e-6  (meters)

# Observation epoch (MJD) — used to compute orbital phase
EPOCH_MJD = 60000.0   # <-- EDIT THIS to match your observation

In [ ]:
# ============================================================
#  BINARY ORBITAL PARAMETERS  (best known values; fitted below)
# ============================================================

# Orbital elements
P_ORB    = 500.0    # days — orbital period             <-- EDIT
T0_MJD   = 59000.0  # MJD  — time of periastron         <-- EDIT
ECC      = 0.0      # eccentricity [0, 1)               <-- EDIT
OMEGA    = 0.0      # argument of periapsis (deg)       <-- EDIT
INC      = 60.0     # inclination (deg), 0=pole-on      <-- EDIT
PA       = 30.0     # position angle of orbit (deg E of N)  <-- EDIT

# Binary geometry
SEP_MAS  = 30.0     # mas — semi-major axis a           <-- EDIT
Q        = 0.3      # mass ratio M_WD / M_giant          <-- EDIT

# Giant star
RPOLE    = 8.0      # mas — polar radius of giant        <-- EDIT
T_GIANT  = 3200.0   # K   — mean surface temperature    <-- EDIT
BETA     = 0.08     # gravity darkening (0.08=convective, 0.25=radiative)

# Limb darkening (quadratic law)
LD_U1    = 0.6      # linear coefficient (cool giant)   <-- EDIT
LD_U2    = 0.1      # quadratic coefficient

# Tessellation resolution
NSIDE    = 4        # 4 → 192 pixels (fast); 8 → 768 pixels (detailed)

---
## 2. Roche Lobe Analysis

Before reconstructing, check how Roche-distorted the giant is and
what the elongation looks like at your orbital phase.

In [ ]:
# Eggleton Roche lobe radius for the giant (primary)
R_L_frac = eggleton_roche_radius(Q)             # R_L / a
R_L_mas  = R_L_frac * SEP_MAS                   # mas
fill_frac = RPOLE / R_L_mas                     # fraction of Roche lobe filled

# Instantaneous separation at observation epoch
D_inst = float(compute_separation(
    a=SEP_MAS, e=ECC, P=P_ORB, T0=T0_MJD,
    tepoch=jnp.array([EPOCH_MJD])
)[0])  # dimensionless d/a
D_mas = D_inst * SEP_MAS

# Async ratio (1.0 = synchronous rotation)
ASYNC = 1.0  # assume synchronous; change if known otherwise

print('=' * 55)
print('  ROCHE LOBE ANALYSIS')
print('=' * 55)
print(f'  Mass ratio q (M_WD/M_giant)    : {Q:.3f}')
print(f'  Eggleton Roche fraction R_L/a  : {R_L_frac:.3f}')
print(f'  Roche lobe radius              : {R_L_mas:.2f} mas')
print(f'  Giant polar radius             : {RPOLE:.2f} mas')
print(f'  Fill factor (rpole/R_L)        : {fill_frac:.3f}')
print()
print(f'  Orbital period                 : {P_ORB:.1f} d')
print(f'  Eccentricity                   : {ECC:.3f}')
print(f'  Separation at epoch            : {D_mas:.2f} mas  (d/a = {D_inst:.4f})')
print()

if fill_frac > 1.0:
    print('  *** ROCHE LOBE OVERFLOW — active mass transfer expected ***')
elif fill_frac > 0.90:
    print('  *** NEAR CONTACT — strong tidal distortion, potential transfer ***')
elif fill_frac > 0.70:
    print('  ** Semi-detached region — measurable tidal elongation **')
else:
    print('  * Detached — weak Roche distortion (may still be detectable)')
print('=' * 55)

In [ ]:
# Visualise how the Roche shape changes with fill factor
tess_vis = tessellation_healpix(n=3)   # coarse, just for plotting

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fills = [0.6, 0.85, 0.98]

for ax, ff in zip(axes, fills):
    rp_ff = ff * R_L_mas          # polar radius that gives this fill fraction
    radii = np.array(compute_roche_shape(
        tess_vis, rp_ff, SEP_MAS, D_inst, Q, ASYNC
    ))

    # Sky projection at current inclination
    rot_mat  = np.array(rotation_matrix(INC, PA, 0.0))
    vxyz_unit = tess_vis.unit_xyz           # (npix, 5, 3)
    vxyz  = radii[:, None, None] * vxyz_unit  # scale by Roche radii
    vxyz_rot = (rot_mat @ vxyz.reshape(-1, 3).T).T.reshape(vxyz.shape)

    centers = vxyz_rot[:, 4, :]             # pixel centres
    vis     = centers[:, 2] > 0             # facing observer

    sc = ax.scatter(centers[vis, 0], centers[vis, 1],
                    c=radii[vis], cmap='plasma',
                    s=80, edgecolors='none')
    plt.colorbar(sc, ax=ax, label='r (mas)')
    ax.set_aspect('equal')
    ax.set_title(f'Fill factor = {ff:.2f}\n(r_pole = {rp_ff:.1f} mas)')
    ax.set_xlabel('ΔRA (mas)')
    ax.set_ylabel('ΔDec (mas)')

    # L1 direction indicator
    ax.axvline(0, ls='--', color='gray', lw=0.8, alpha=0.5)
    ax.axhline(0, ls='--', color='gray', lw=0.8, alpha=0.5)

fig.suptitle(f'Roche geometry (q={Q}, i={INC}°, PA={PA}°)', fontsize=13)
plt.tight_layout()
plt.savefig('roche_shape_comparison.png', dpi=150)
print('Saved: roche_shape_comparison.png')
plt.show()

---
## 3. Load Interferometric Data

Load your OIFITS file, or generate synthetic data to test the pipeline.

In [ ]:
def make_synthetic_data(n_v2=80, n_t3=40, rpole=8.0, sep=30.0,
                        q=0.3, D=1.0, inc=60.0, pa=30.0,
                        wave=1.65e-6, noise_v2=0.02, noise_t3=5.0,
                        seed=42):
    """Generate synthetic OIFITS-like data for a Roche-distorted giant.

    Use this to test the pipeline before you have real observations.
    Replace with read_oifits(OIFITS_FILE) for real data.
    """
    rng = np.random.default_rng(seed)

    # Random UV coverage (CHARA-like baselines 50-330 m)
    b_len = rng.uniform(50, 330, n_v2 + n_t3 * 2)
    b_ang = rng.uniform(0, np.pi, len(b_len))
    u_all = (b_len * np.cos(b_ang)) / wave    # cycles/radian
    v_all = (b_len * np.sin(b_ang)) / wave

    nuv = len(u_all)
    uv  = jnp.array(np.vstack([u_all, v_all]))
    wl  = jnp.full(nuv, wave)

    indx_v2 = jnp.arange(n_v2)
    t3_1 = jnp.arange(n_t3)
    t3_2 = jnp.arange(n_t3) + n_v2
    t3_3 = jnp.arange(n_t3) + n_v2 + n_t3

    # Build a Roche star and compute 'truth' observables
    tess_s  = tessellation_healpix(n=3)
    radii_s = np.array(compute_roche_shape(tess_s, rpole, sep, D, q, 1.0))

    rot_s   = np.array(rotation_matrix(inc, pa, 0.0))
    vxyz_s  = radii_s[:, None, None] * tess_s.unit_xyz
    vxyz_r  = (rot_s @ vxyz_s.reshape(-1, 3).T).T.reshape(vxyz_s.shape)
    vis_s   = vxyz_r[:, 4, 2] > 0
    eff_diam = 2.0 * float(radii_s.max())

    star_s  = Star(
        tess=tess_s,
        theta=jnp.array(tess_s.unit_spherical[:, 4, 1]),
        phi=jnp.array(tess_s.unit_spherical[:, 4, 2]),
        x=jnp.array(vxyz_r[:, 4, 0]),
        y=jnp.array(vxyz_r[:, 4, 1]),
        z=jnp.array(vxyz_r[:, 4, 2]),
        visible=jnp.array(vis_s),
        intensities=jnp.ones(tess_s.npix),
        diameter=eff_diam,
        inclination=inc,
        orientation=pa,
    )

    proto_data = OIData(
        v2=jnp.zeros(n_v2), v2_err=jnp.ones(n_v2)*noise_v2, nv2=n_v2,
        t3phi=jnp.zeros(n_t3), t3phi_err=jnp.ones(n_t3)*noise_t3, nt3=n_t3,
        t3amp=jnp.zeros(n_t3), t3amp_err=jnp.ones(n_t3),
        uv=uv, nuv=nuv,
        indx_v2=indx_v2, indx_t3_1=t3_1, indx_t3_2=t3_2, indx_t3_3=t3_3,
        wavelengths=wl,
        mean_mjd=EPOCH_MJD, filename='synthetic',
    )

    # Compute truth observables (polyft computed on-the-fly inside compute_observables)
    geom_s = star_s.to_geometry()
    v2_true, _, t3phi_true = compute_observables(star_s.intensities, geom_s, proto_data)

    # Add noise
    v2_noisy   = np.array(v2_true)    + rng.normal(0, noise_v2, n_v2)
    t3_noisy   = np.array(t3phi_true) + rng.normal(0, noise_t3, n_t3)

    return OIData(
        v2=jnp.array(np.clip(v2_noisy, 0, 1)),
        v2_err=jnp.ones(n_v2) * noise_v2,
        nv2=n_v2,
        t3phi=jnp.array(t3_noisy),
        t3phi_err=jnp.ones(n_t3) * noise_t3,
        nt3=n_t3,
        t3amp=jnp.zeros(n_t3),
        t3amp_err=jnp.ones(n_t3),
        uv=uv, nuv=nuv,
        indx_v2=indx_v2, indx_t3_1=t3_1, indx_t3_2=t3_2, indx_t3_3=t3_3,
        wavelengths=wl,
        mean_mjd=EPOCH_MJD, filename='synthetic',
    )


# ----- Load real data or fall back to synthetic -----
oifits_path = Path(OIFITS_FILE)

if oifits_path.exists():
    print(f'Loading real data: {OIFITS_FILE}')
    oi_data = read_oifits(str(oifits_path))
    USING_SYNTHETIC = False
else:
    print(f'File not found: {OIFITS_FILE}')
    print('=> Generating synthetic data for demonstration')
    oi_data = make_synthetic_data(
        rpole=RPOLE, sep=SEP_MAS, q=Q, D=D_inst,
        inc=INC, pa=PA, wave=1.65e-6
    )
    USING_SYNTHETIC = True

print()
print(f'  V2  measurements : {oi_data.nv2}')
print(f'  T3phi measurements : {oi_data.nt3}')
print(f'  UV  points       : {oi_data.nuv}')
print(f'  Epoch (MJD)      : {oi_data.mean_mjd:.1f}')
wl_um = np.array(oi_data.wavelengths) * 1e6
print(f'  Wavelengths      : {wl_um.min():.2f} - {wl_um.max():.2f} um')

In [ ]:
# Quick look at the data
u = np.array(oi_data.uv[0, :])
v = np.array(oi_data.uv[1, :])
bl_Mrad = np.sqrt(u**2 + v**2) * 1e-6   # baseline in Mrad^-1

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# UV coverage
ax = axes[0]
ax.scatter(u * 1e-6, v * 1e-6, s=8, alpha=0.6, color='steelblue')
ax.scatter(-u * 1e-6, -v * 1e-6, s=8, alpha=0.6, color='steelblue')
ax.set_xlabel('u (M rad$^{-1}$)')
ax.set_ylabel('v (M rad$^{-1}$)')
ax.set_title('UV coverage')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

# V² vs baseline
ax = axes[1]
v2_vals = np.array(oi_data.v2)
v2_err  = np.array(oi_data.v2_err)
bl_v2   = bl_Mrad[np.array(oi_data.indx_v2)]
ax.errorbar(bl_v2, v2_vals, yerr=v2_err,
            fmt='.', color='steelblue', ecolor='gray', alpha=0.7)
ax.set_xlabel('Baseline (M rad$^{-1}$)')
ax.set_ylabel('V²')
ax.set_title('Squared visibilities')
ax.set_ylim(-0.05, 1.1)
ax.grid(True, alpha=0.3)

# Closure phases
ax = axes[2]
t3_vals = np.array(oi_data.t3phi)
t3_err  = np.array(oi_data.t3phi_err)
bl_t3   = bl_Mrad[np.array(oi_data.indx_t3_1)]
ax.errorbar(bl_t3, t3_vals, yerr=t3_err,
            fmt='.', color='darkorange', ecolor='gray', alpha=0.7)
ax.axhline(0, color='k', lw=0.8, ls='--')
ax.set_xlabel('Baseline 1 (M rad$^{-1}$)')
ax.set_ylabel('Closure phase (deg)')
ax.set_title('Closure phases')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('data_overview.png', dpi=150)
print('Saved: data_overview.png')
plt.show()

---
## 4. Build Roche-Distorted Star Geometry

This creates a `Star` object with the Roche-elongated shape of the giant.
The key function is `compute_roche_shape()` which returns r(θ,φ) at each
tessellation point, then we scale the unit sphere vertices accordingly.

In [ ]:
def create_roche_star(tess, rpole, sep, D_dimensionless, q,
                      inclination, position_angle, intensities,
                      async_ratio=1.0, ld_u1=0.0, ld_u2=0.0):
    """Create a Star with Roche-distorted geometry.

    The star's shape is r(theta, phi) = Roche equipotential that
    passes through the pole at radius `rpole`.

    Args:
        tess            : HEALPix tessellation
        rpole           : Polar radius (mas)
        sep             : Binary semi-major axis a (mas)
        D_dimensionless : Instantaneous separation d/a (use 1.0 for circular)
        q               : Mass ratio M_companion / M_giant
        inclination     : Inclination (deg)
        position_angle  : PA of orbit on sky (deg)
        intensities     : (npix,) surface intensity map
        async_ratio     : omega_rot / omega_orb (1.0 = synchronous)
        ld_u1, ld_u2    : Quadratic limb darkening coefficients

    Returns:
        Star object ready for reconstruct_stellar_surface()
    """
    # Roche radii at every pixel centre  (npix,)
    radii = np.array(compute_roche_shape(
        tess, rpole, sep, D_dimensionless, q, async_ratio
    ))

    # Scale unit-sphere vertices by pixel-specific radius
    # unit_xyz: (npix, 5, 3)  →  vertices_xyz: (npix, 5, 3)
    vertices_xyz = radii[:, None, None] * tess.unit_xyz   # broadcast

    # Rotate to observer frame
    rot_mat = np.array(rotation_matrix(inclination, position_angle, 0.0))
    vxyz_flat    = vertices_xyz.reshape(-1, 3)
    vxyz_rot_flat = (rot_mat @ vxyz_flat.T).T
    vxyz_rot = vxyz_rot_flat.reshape(vertices_xyz.shape)

    # Visibility mask (pixel centre z > 0)
    vis = jnp.array(vxyz_rot[:, 4, 2] > 0)

    # Pixel centres on sky
    centers = vxyz_rot[:, 4, :]          # (npix, 3)
    x = jnp.array(centers[:, 0])
    y = jnp.array(centers[:, 1])
    z = jnp.array(centers[:, 2])

    # Effective diameter = 2 * max projected extent
    eff_diameter = 2.0 * float(radii.max())

    # Limb darkening weights
    ld_coeffs = jnp.array([ld_u1, ld_u2])

    theta = jnp.array(tess.unit_spherical[:, 4, 1])   # colatitude
    phi   = jnp.array(tess.unit_spherical[:, 4, 2])   # longitude

    return Star(
        tess=tess, theta=theta, phi=phi,
        x=x, y=y, z=z,
        visible=vis,
        intensities=jnp.array(intensities),
        diameter=eff_diameter,
        inclination=inclination,
        orientation=position_angle,
        ld_coeffs=ld_coeffs,
    )


print('create_roche_star() defined')

In [ ]:
# Build tessellation and star
tess = tessellation_healpix(n=NSIDE)
print(f'Tessellation: nside={tess.nside}, npix={tess.npix}')

# Initial intensity map — uniform at T_GIANT
x0 = jnp.ones(tess.npix) * T_GIANT

star = create_roche_star(
    tess        = tess,
    rpole       = RPOLE,
    sep         = SEP_MAS,
    D_dimensionless = D_inst,
    q           = Q,
    inclination = INC,
    position_angle = PA,
    intensities = x0 / T_GIANT,   # normalised
    ld_u1       = LD_U1,
    ld_u2       = LD_U2,
)

n_vis = int(jnp.sum(star.visible))
print(f'Star: diameter={star.diameter:.2f} mas, {n_vis}/{tess.npix} pixels visible')
print(f'Fill factor check: rpole={RPOLE:.2f} mas, R_L={R_L_mas:.2f} mas → f={RPOLE/R_L_mas:.3f}')

In [ ]:
# Show sky-projected Roche star vs same star as sphere
star_sph = create_star(
    tess=tess, inclination=INC, orientation=PA,
    intensities=jnp.ones(tess.npix),
    diameter=2*RPOLE,
)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))

for ax, st, title in zip(
    axes,
    [star_sph, star],
    ['Sphere (r_pole)', f'Roche distorted (fill={RPOLE/R_L_mas:.2f})'],
):
    vis = np.array(st.visible)
    x_v = np.array(st.x)[vis]
    y_v = np.array(st.y)[vis]
    z_v = np.array(st.z)[vis]
    sc = ax.scatter(x_v, y_v, c=z_v, cmap='RdYlBu_r', s=60, edgecolors='none')
    plt.colorbar(sc, ax=ax, label='z (toward observer)')
    ax.set_aspect('equal')
    ax.set_title(title)
    ax.set_xlabel('ΔRA (mas)')
    ax.set_ylabel('ΔDec (mas)')
    ax.grid(True, alpha=0.2)

# Mark L1 direction
for ax in axes:
    ax.axvline(0, ls=':', color='gray', lw=0.8)
    ax.axhline(0, ls=':', color='gray', lw=0.8)

plt.suptitle(f'Roche elongation (q={Q}, i={INC}°, PA={PA}°)', fontsize=13)
plt.tight_layout()
plt.savefig('roche_star_sky.png', dpi=150)
print('Saved: roche_star_sky.png')
plt.show()

---
## 5. Surface Brightness Reconstruction

With the Roche geometry fixed, reconstruct the temperature/brightness map
of the giant's surface.  
Run Section 4 → Parameter Fitting first if you want to fit geometry simultaneously.

In [ ]:
# Regularization — tune weights to balance smoothness vs data fit
# Rule of thumb: start with MEM~0.05, reduce if chi2_red >> 1
#               increase if map is too noisy
regularizers = [
    {'type': 'mem', 'weight': 0.03},    # maximum entropy (smoothness)
    {'type': 'tv',  'weight': 0.005},   # total variation (edges)
]

# Temperature bounds for reconstruction
T_MIN = 2000.0    # K — minimum surface temperature  <-- EDIT if needed
T_MAX = 4500.0    # K — maximum surface temperature  <-- EDIT if needed
MAXITER = 200

print(f'Regularizers: {regularizers}')
print(f'T range: {T_MIN} – {T_MAX} K')

In [ ]:
print('=' * 70)
print('  IMAGE RECONSTRUCTION — ROCHE GIANT')
print('=' * 70)

result = reconstruct_stellar_surface(
    oi_data     = oi_data,
    star        = star,
    x_start     = x0,
    regularizers = regularizers,
    bounds      = (T_MIN, T_MAX),
    maxiter     = MAXITER,
    verbose     = True,
)

n_obs = oi_data.nv2 + oi_data.nt3
chi2_red = result.chi2_final / n_obs

print()
print(f'  Success      : {result.success}')
print(f'  Iterations   : {result.iterations}')
print(f'  Final χ²     : {result.chi2_final:.2f}')
print(f'  Reduced χ²   : {chi2_red:.3f}  (ideal ~ 1.0)')
print(f'  Time         : {result.time_elapsed:.1f} s')

T_map = result.x_solution
print()
print(f'  Temperature map: {float(T_map.min()):.0f} – {float(T_map.max()):.0f} K')
print(f'  Mean: {float(T_map.mean()):.0f} K')
print(f'  Contrast: {100*(float(T_map.max())-float(T_map.min()))/float(T_map.mean()):.1f}%')

In [ ]:
fig = plt.figure(figsize=(16, 5))
gs  = gridspec.GridSpec(1, 4, figure=fig)

# --- Sky image ---
ax1 = fig.add_subplot(gs[0])
vis = np.array(star.visible)
sc = ax1.scatter(
    np.array(star.x)[vis], np.array(star.y)[vis],
    c=np.array(T_map)[vis], cmap='hot',
    vmin=T_MIN, vmax=T_MAX, s=80, edgecolors='none'
)
plt.colorbar(sc, ax=ax1, label='T (K)')
ax1.set_aspect('equal')
ax1.set_title(f'Roche giant (sky)\n{RPOLE:.1f} mas polar radius')
ax1.set_xlabel('ΔRA (mas)'); ax1.set_ylabel('ΔDec (mas)')
ax1.grid(True, alpha=0.2)

# --- Mollweide surface map ---
ax2 = fig.add_subplot(gs[1], projection='mollweide')
lon = np.array(star.phi) - np.pi           # [-pi, pi]
lat = np.pi/2 - np.array(star.theta)       # [-pi/2, pi/2]
sc2 = ax2.scatter(lon, lat, c=np.array(T_map), s=60,
                  cmap='hot', vmin=T_MIN, vmax=T_MAX,
                  edgecolors='none')
plt.colorbar(sc2, ax=ax2, orientation='horizontal', pad=0.05, fraction=0.046, label='T (K)')
ax2.set_title('Surface map (Mollweide)')
ax2.grid(True, alpha=0.3)

# --- Convergence ---
ax3 = fig.add_subplot(gs[2])
iters = range(len(result.history['f_total']))
ax3.semilogy(iters, result.history['chi2'],   'r-', label='χ²',  alpha=0.8)
ax3.semilogy(iters, result.history['reg'],    'b-', label='Reg', alpha=0.8)
ax3.semilogy(iters, result.history['f_total'],'k-', label='Total', lw=2)
ax3.set_xlabel('Function evaluations')
ax3.set_ylabel('Objective')
ax3.set_title('Convergence')
ax3.legend(fontsize=9); ax3.grid(True, alpha=0.3)

# --- Temperature histogram ---
ax4 = fig.add_subplot(gs[3])
ax4.hist(np.array(T_map), bins=25, color='steelblue', edgecolor='white', alpha=0.8)
ax4.axvline(T_GIANT, color='r', ls='--', label=f'T_eff = {T_GIANT:.0f} K')
ax4.set_xlabel('Temperature (K)')
ax4.set_ylabel('Pixels')
ax4.set_title('T distribution')
ax4.legend(fontsize=9); ax4.grid(True, alpha=0.3)

plt.suptitle(f'Reconstruction  χ²_red = {chi2_red:.3f}', fontsize=13)
plt.tight_layout()
plt.savefig('reconstruction_result.png', dpi=150)
print('Saved: reconstruction_result.png')
plt.show()

---
## 6. Binary Parameter Fitting

Fit the binary/Roche parameters by minimising χ² while holding the
surface map fixed.  Three stages:

1. **1-D χ² profiles** — see how sensitive the data are to each parameter
2. **2-D grid scan** — find the global minimum region (q vs inclination)
3. **Full optimisation** — gradient-free Nelder-Mead from the best grid point

For a full joint fit (geometry + image simultaneously) see Section 7.

In [ ]:
def chi2_roche_model(
    q, rpole, sep, D_dim, inclination, pa,
    oi_data, tess, intensity_map=None
):
    """Compute chi2 for a given set of Roche binary parameters.

    Args:
        q           : Mass ratio M_WD / M_giant
        rpole       : Polar radius (mas)
        sep         : Semi-major axis a (mas)
        D_dim       : Dimensionless separation d/a
        inclination : Inclination (deg)
        pa          : Position angle (deg)
        oi_data     : OIData
        tess        : Tessellation
        intensity_map : (npix,) or None  (None = uniform disk)

    Returns:
        chi2_total : scalar float
        chi2_v2    : V2 contribution
        chi2_t3    : T3phi contribution
    """
    npix = tess.npix
    if intensity_map is None:
        intensities = jnp.ones(npix)
    else:
        intensities = jnp.array(intensity_map)
    intensities = intensities / float(intensities.mean())

    try:
        st = create_roche_star(
            tess=tess, rpole=rpole, sep=sep,
            D_dimensionless=D_dim, q=q,
            inclination=inclination, position_angle=pa,
            intensities=intensities,
        )

        geom = st.to_geometry()

        # compute_chi2 with return_components=True returns
        # (total, chi2_v2, chi2_t3amp, chi2_t3phi)
        total_chi2, c_v2, c_t3amp, c_t3phi = compute_chi2(
            intensities, geom, oi_data, return_components=True
        )
        return float(total_chi2), float(c_v2), float(c_t3amp + c_t3phi)

    except Exception as e:
        return 1e12, 1e12, 1e12


# Quick sanity check on the reference model
c2, c2_v2, c2_t3 = chi2_roche_model(
    Q, RPOLE, SEP_MAS, D_inst, INC, PA, oi_data, tess,
    intensity_map=T_map
)
n_obs = oi_data.nv2 + oi_data.nt3
print(f'Reference model: chi2 = {c2:.2f}, chi2_red = {c2/n_obs:.3f}')
print(f'  V2 contribution: {c2_v2:.2f}   T3phi contribution: {c2_t3:.2f}')

In [ ]:
# ---- 1-D chi2 profiles ----
# Show sensitivity to each key parameter around the reference values

params_scan = {
    'q (mass ratio)':       (np.linspace(0.1, 0.9, 20),  Q,     'Q'),
    'rpole (mas)':          (np.linspace(0.4*R_L_mas,
                                         0.98*R_L_mas, 20), RPOLE, 'RPOLE'),
    'inclination (deg)':    (np.linspace(20, 85, 20),     INC,   'INC'),
    'PA (deg)':             (np.linspace(0, 180, 20),     PA,    'PA'),
}

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, (label, (grid, ref, _)) in zip(axes.flat, params_scan.items()):
    chi2s = []
    for val in grid:
        kw = dict(q=Q, rpole=RPOLE, sep=SEP_MAS, D_dim=D_inst,
                  inclination=INC, pa=PA)
        if label.startswith('q'):          kw['q']           = val
        elif label.startswith('rp'):       kw['rpole']       = val
        elif label.startswith('i'):        kw['inclination'] = val
        elif label.startswith('P'):        kw['pa']          = val

        c2, _, _ = chi2_roche_model(**kw,
            oi_data=oi_data, tess=tess, intensity_map=T_map)
        chi2s.append(c2 / n_obs)

    ax.plot(grid, chi2s, 'o-', color='steelblue', ms=5)
    ax.axvline(ref, color='r', ls='--', label=f'ref = {ref:.2f}')
    ax.axhline(1.0, color='gray', ls=':', lw=0.8, label='χ²_red = 1')
    ax.set_xlabel(label)
    ax.set_ylabel('χ²_red')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('χ² sensitivity to binary parameters', fontsize=13)
plt.tight_layout()
plt.savefig('chi2_profiles.png', dpi=150)
print('Saved: chi2_profiles.png')
plt.show()

In [ ]:
# ---- 2-D grid scan: q vs inclination ----
# The most degenerate pair for Roche fitting — map the chi2 surface

q_grid   = np.linspace(0.15, 0.75, 18)
inc_grid = np.linspace(25, 80, 18)

chi2_grid = np.full((len(inc_grid), len(q_grid)), np.nan)

print(f'Running {len(q_grid) * len(inc_grid)} model evaluations...')
for ii, inc_val in enumerate(inc_grid):
    for iq, q_val in enumerate(q_grid):
        c2, _, _ = chi2_roche_model(
            q=q_val, rpole=RPOLE, sep=SEP_MAS, D_dim=D_inst,
            inclination=inc_val, pa=PA,
            oi_data=oi_data, tess=tess, intensity_map=T_map
        )
        chi2_grid[ii, iq] = c2 / n_obs

# Best-fit location
ii_best, iq_best = np.unravel_index(np.nanargmin(chi2_grid), chi2_grid.shape)
q_best   = q_grid[iq_best]
inc_best = inc_grid[ii_best]
chi2_best = chi2_grid[ii_best, iq_best]

print(f'Best grid point: q={q_best:.3f}, i={inc_best:.1f}°, χ²_red={chi2_best:.3f}')

fig, ax = plt.subplots(figsize=(8, 6))
pm = ax.pcolormesh(q_grid, inc_grid, chi2_grid,
                   cmap='viridis_r',
                   vmax=min(chi2_best * 3, np.nanpercentile(chi2_grid, 90)))
plt.colorbar(pm, ax=ax, label='χ²_red')
# Δchi2 contours
cs = ax.contour(q_grid, inc_grid, chi2_grid,
                levels=[chi2_best + dchi2/n_obs for dchi2 in [2.3, 6.17]],
                colors=['white', 'yellow'], linestyles='--')
ax.clabel(cs, fmt='Δχ²=%.1f', fontsize=8)
ax.scatter([q_best], [inc_best], marker='*', s=200, color='red', zorder=5,
           label=f'Best: q={q_best:.3f}, i={inc_best:.1f}°')
ax.scatter([Q], [INC], marker='+', s=150, color='lime', zorder=5,
           label=f'Input: q={Q:.3f}, i={INC:.1f}°')
ax.set_xlabel('Mass ratio q = M_WD / M_giant')
ax.set_ylabel('Inclination (deg)')
ax.set_title('χ² grid scan: mass ratio vs inclination')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('chi2_grid_q_inc.png', dpi=150)
print('Saved: chi2_grid_q_inc.png')
plt.show()

In [ ]:
# ---- Full Nelder-Mead optimisation from best grid point ----
# Parameters to fit: [q, rpole, inclination, PA]
# sep (a) is usually constrained by distance + spectroscopy — keep fixed
# eccentricity: add 'e' to the list if your system is eccentric

def objective(params):
    """Objective function for scipy.optimize.minimize."""
    q_p, rpole_p, inc_p, pa_p = params

    # Bounds enforcement via penalty
    R_L_p = float(eggleton_roche_radius(q_p)) * SEP_MAS
    if q_p < 0.05 or q_p > 2.0:
        return 1e10
    if rpole_p < 0.5 or rpole_p > 0.999 * R_L_p:
        return 1e10
    if inc_p < 5 or inc_p > 85:
        return 1e10

    D_p = float(compute_separation(
        a=SEP_MAS, e=ECC, P=P_ORB, T0=T0_MJD,
        tepoch=jnp.array([EPOCH_MJD])
    )[0])

    c2, _, _ = chi2_roche_model(
        q=q_p, rpole=rpole_p, sep=SEP_MAS, D_dim=D_p,
        inclination=inc_p, pa=pa_p,
        oi_data=oi_data, tess=tess, intensity_map=T_map
    )
    return c2


# Starting point from grid scan
x0_fit = [q_best, RPOLE, inc_best, PA]
print(f'Starting from: q={q_best:.3f}, rpole={RPOLE:.2f}, inc={inc_best:.1f}°, PA={PA:.1f}°')

opt = so.minimize(
    objective, x0_fit,
    method='Nelder-Mead',
    options={'maxiter': 500, 'xatol': 1e-4, 'fatol': 0.1, 'disp': True},
)

q_fit, rpole_fit, inc_fit, pa_fit = opt.x

print()
print('=' * 50)
print('  BEST-FIT BINARY PARAMETERS')
print('=' * 50)
print(f'  q   (M_WD/M_giant) : {q_fit:.4f}  (input: {Q:.4f})')
print(f'  r_pole             : {rpole_fit:.3f} mas  (input: {RPOLE:.3f})')
print(f'  inclination        : {inc_fit:.2f} deg  (input: {INC:.2f})')
print(f'  position angle     : {pa_fit:.2f} deg  (input: {PA:.2f})')
print(f'  χ²                 : {opt.fun:.2f}  (χ²_red = {opt.fun/n_obs:.3f})')
print(f'  Converged: {opt.success}  ({opt.message})')

R_L_fit = float(eggleton_roche_radius(q_fit)) * SEP_MAS
print(f'  Fill factor        : {rpole_fit/R_L_fit:.3f}')
print('=' * 50)

In [ ]:
# Rebuild star with best-fit parameters and re-run reconstruction

print('Rebuilding star with best-fit parameters...')
D_fit = float(compute_separation(
    a=SEP_MAS, e=ECC, P=P_ORB, T0=T0_MJD,
    tepoch=jnp.array([EPOCH_MJD])
)[0])

star_fit = create_roche_star(
    tess=tess, rpole=rpole_fit, sep=SEP_MAS,
    D_dimensionless=D_fit, q=q_fit,
    inclination=inc_fit, position_angle=pa_fit,
    intensities=T_map / float(T_map.mean()),
    ld_u1=LD_U1, ld_u2=LD_U2,
)

print('Re-running reconstruction with best-fit geometry...')
result_fit = reconstruct_stellar_surface(
    oi_data=oi_data,
    star=star_fit,
    x_start=T_map,
    regularizers=regularizers,
    bounds=(T_MIN, T_MAX),
    maxiter=MAXITER,
    verbose=False,
)

T_map_fit = result_fit.x_solution
chi2_red_fit = result_fit.chi2_final / n_obs

print(f'\nFinal χ²_red (best-fit geometry): {chi2_red_fit:.4f}')
print(f'Improvement: {(chi2_red - chi2_red_fit)/chi2_red * 100:.1f}%')

---
## 7. Multi-Epoch Fitting  *(optional)*

If you have OIFITS data at multiple orbital phases, you can constrain
the binary parameters much better by fitting all phases simultaneously.
A single surface map is reconstructed while the viewing angle at each
epoch follows the Keplerian orbit.

In [ ]:
# ============================================================
#  MULTI-EPOCH DATA  — edit this list
# ============================================================
MULTI_EPOCH_FILES = [
    # ('path/to/epoch1.fits', 60000.0),   # (filename, MJD)
    # ('path/to/epoch2.fits', 60250.0),
    # ('path/to/epoch3.fits', 60500.0),
]

if not MULTI_EPOCH_FILES:
    print('No multi-epoch files specified — skipping this section.')
    print('Add (filename, MJD) pairs to MULTI_EPOCH_FILES above to use.')

In [ ]:
from rotir_jax.reconstruction.multi_epoch import Epoch, reconstruct_multi_epoch

if MULTI_EPOCH_FILES:
    epochs = []
    for fname, mjd in MULTI_EPOCH_FILES:
        data = read_oifits(fname)

        # Orbital phase at this epoch
        nu = float(compute_true_anomaly(P_ORB, ECC, T0_MJD, jnp.array([mjd]))[0])
        D_e = float(compute_separation(SEP_MAS, ECC, P_ORB, T0_MJD, jnp.array([mjd]))[0])

        # Build star at this orbital phase
        # orbital longitude of secondary relative to primary = nu + omega
        # rotation phase of giant = synchronous → rot_phase = orbital_phase
        # (add dP correction here if period derivative is known)

        star_e = create_roche_star(
            tess=tess, rpole=rpole_fit, sep=SEP_MAS,
            D_dimensionless=D_e, q=q_fit,
            inclination=inc_fit, position_angle=pa_fit,
            intensities=jnp.ones(tess.npix),
        )
        orbital_phase = ((mjd - T0_MJD) % P_ORB) / P_ORB
        epochs.append(Epoch(
            oi_data=data,
            rotation_phase=orbital_phase,   # synchronous rotation
            orbital_phase=orbital_phase,
            mjd=mjd,
        ))
        print(f'  Epoch {mjd:.0f}: orbital phase = {orbital_phase:.3f}')

    print(f'\nRunning multi-epoch reconstruction ({len(epochs)} epochs)...')
    result_multi = reconstruct_multi_epoch(
        epochs=epochs,
        star=star_e,   # template star (geometry updated per epoch internally)
        mode='static',
        regularizers=regularizers,
        maxiter=200, verbose=True,
    )
    T_map_multi = result_multi.x_solution
    print(f'Multi-epoch χ²_red = {result_multi.chi2_final / sum(e.oi_data.nv2+e.oi_data.nt3 for e in epochs):.3f}')

---
## 8. Results Summary & Export

In [ ]:
# Final comparison: initial vs best-fit geometry
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (T, st, lbl) in zip(axes, [
    (T_map,     star,     f'Initial geometry\nq={Q:.3f}, i={INC:.1f}°, f={RPOLE/R_L_mas:.3f}'),
    (T_map_fit, star_fit, f'Best-fit geometry\nq={q_fit:.3f}, i={inc_fit:.1f}°, f={rpole_fit/R_L_fit:.3f}'),
]):
    vis = np.array(st.visible)
    sc = ax.scatter(
        np.array(st.x)[vis], np.array(st.y)[vis],
        c=np.array(T)[vis], cmap='hot',
        vmin=T_MIN, vmax=T_MAX, s=80, edgecolors='none'
    )
    plt.colorbar(sc, ax=ax, label='T (K)')
    ax.set_aspect('equal')
    ax.set_title(lbl)
    ax.set_xlabel('ΔRA (mas)'); ax.set_ylabel('ΔDec (mas)')
    ax.grid(True, alpha=0.2)

plt.suptitle('Surface temperature maps — comparison', fontsize=13)
plt.tight_layout()
plt.savefig('final_comparison.png', dpi=150)
print('Saved: final_comparison.png')
plt.show()

In [ ]:
# Save results
np.save('T_map_initial_geom.npy',  np.array(T_map))
np.save('T_map_bestfit_geom.npy',  np.array(T_map_fit))
np.save('tessellation_theta.npy',  np.array(star.theta))
np.save('tessellation_phi.npy',    np.array(star.phi))

summary = {
    'Input parameters': {
        'q': Q, 'rpole_mas': RPOLE, 'sep_mas': SEP_MAS,
        'inclination_deg': INC, 'PA_deg': PA, 'fill_factor': RPOLE/R_L_mas,
    },
    'Best-fit parameters': {
        'q': round(q_fit, 4),
        'rpole_mas': round(rpole_fit, 3),
        'sep_mas': SEP_MAS,
        'inclination_deg': round(inc_fit, 2),
        'PA_deg': round(pa_fit, 2),
        'fill_factor': round(rpole_fit / R_L_fit, 3),
        'chi2_red': round(chi2_red_fit, 4),
    },
    'Temperature map (initial geom)': {
        'T_min_K': round(float(T_map.min()), 0),
        'T_max_K': round(float(T_map.max()), 0),
        'T_mean_K': round(float(T_map.mean()), 0),
        'contrast_pct': round(100*(float(T_map.max())-float(T_map.min()))/float(T_map.mean()), 1),
    },
    'Temperature map (best-fit geom)': {
        'T_min_K': round(float(T_map_fit.min()), 0),
        'T_max_K': round(float(T_map_fit.max()), 0),
        'T_mean_K': round(float(T_map_fit.mean()), 0),
        'contrast_pct': round(100*(float(T_map_fit.max())-float(T_map_fit.min()))/float(T_map_fit.mean()), 1),
    },
}

import json
with open('symbiotic_star_results.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('Results saved:')
print('  T_map_initial_geom.npy')
print('  T_map_bestfit_geom.npy')
print('  tessellation_theta/phi.npy')
print('  symbiotic_star_results.json')
print()
print(json.dumps(summary, indent=2))

---
## Tips and troubleshooting

### χ²_red >> 1 (bad fit)
- Reduce regularization weights (try `mem` × 0.1)
- Check that `SEP_MAS` and `RPOLE` are sensible (star must be partially resolved)
- Verify `EPOCH_MJD` matches the observation date
- Try larger `NSIDE` (8 or 16) for better angular resolution

### χ²_red << 1 (over-fit)
- Increase regularization weights
- Check error bars in your OIFITS file — they may be underestimated

### Roche shape looks wrong
- The L1 point is always toward the white dwarf companion (positive x-axis)
- If the elongation points the wrong way, try `PA + 90`
- For near-contact systems (fill > 0.95), Newton-Raphson may need more iterations —
  edit `max_iter` in `compute_roche_shape()`

### Parameter fitting converges to wrong minimum
- Run a finer grid scan first (increase grid resolution)
- Fix `PA` from imaging and fit only `q`, `rpole`, `inclination`
- Use spectroscopic mass function to constrain `q × sin(i)³`

### Want uncertainties?
- The χ²_red = 1 contours on the 2-D grid give 68% confidence intervals
- For full posterior samples: see the Bayesian inference in `test_bayesian.py`
  using `BayesianStellarModel` with `run_nuts()` or `run_mgvi()`

### Wavelength dependence
- Cool giants are much larger at TiO bands (~700 nm) than in the near-IR
- Use `WAVE_MIN` / `WAVE_MAX` to select a narrow wavelength window
- Run the notebook separately for each wavelength band to map chromatic variation

---

**Reference:** Aufdenberg et al. (2021), ApJ 920, 130 — SPICA Roche model  
**Code:** ROTIR-JAX — Python/JAX port of ROTIR  